# Mid-term model temperature reconstruction

In [ ]:
# @title Currently, only datasets in the following form are supported.

import pandas as pd
pd.DataFrame({'date': ['dd.mm.rrrr'],
              'time':['hh'],
              'measurements':['floating number'],
              'weather_station':['floating number']})

In [2]:
# @title Install packages
%%capture
!pip install scikit-fda

In [3]:
# @title Prepare environment

import pandas as pd
import numpy as np
import skfda
import matplotlib.pyplot as plt
from google.colab import files

np.float_ = np.float64

def transformation_of_data(temp_df, n):

    temp_df = temp_df.copy()
    temp_df.loc[:,'date'] = pd.to_datetime(temp_df['date'])

    start_date = min(temp_df['date'])

    temp_df.loc[:,'rep'] = (temp_df['date'] - start_date).dt.days + 1
    temp_df.loc[:,'hour'] = temp_df['date'].dt.hour
    temp_df = temp_df.groupby(['rep', 'hour']).agg({'Y': 'mean', 'X': 'mean'}).reset_index()
    group_counts = temp_df.groupby('rep').size()
    valid_groups = group_counts[group_counts == 24].index
    df_filtered = temp_df[temp_df['rep'].isin(valid_groups)].copy()
    df_filtered.loc[:,'idx'] = df_filtered.groupby('rep').cumcount() #+ 1
    temp = df_filtered.pivot(index = 'idx', columns='rep', values='Y')
    weather = df_filtered.pivot(index = 'idx', columns='rep', values='X')

    all_sets = {'weather': weather, 'temp': temp}
    train_sets = {'weather': weather.iloc[:, n:], 'temp': temp.iloc[:, n:]}
    test_sets = {'weather': weather.iloc[:, :n], 'temp': temp.iloc[:, :n]}
    return all_sets, train_sets, test_sets, start_date

def transformation_of_data_2(temp_df):

    temp_df = temp_df.copy()
    temp_df.loc[:,'date'] = pd.to_datetime(temp_df['date'])

    start_date = min(temp_df['date'])

    temp_df.loc[:,'rep'] = (temp_df['date'] - start_date).dt.days + 1
    temp_df.loc[:,'hour'] = temp_df['date'].dt.hour
    temp_df = temp_df.groupby(['rep', 'hour']).agg({'X': 'mean'}).reset_index()
    group_counts = temp_df.groupby('rep').size()
    valid_groups = group_counts[group_counts == 24].index
    df_filtered = temp_df[temp_df['rep'].isin(valid_groups)].copy()
    df_filtered.loc[:,'idx'] = df_filtered.groupby('rep').cumcount() #+ 1
    weather = df_filtered.pivot(index = 'idx', columns='rep', values='X')

    results = {'weather': weather}
    return results, start_date

def reconstruction_mid_term(train_set):
    t = np.linspace(0, 23, 24)
    fourier_basis = skfda.representation.basis.FourierBasis(domain_range=(0,23), n_basis=3, period=24)

    Y_train = skfda.FDataGrid(
        data_matrix=train_set['temp'].T,
        grid_points=t,
    ).to_basis(fourier_basis)
    X_train = skfda.FDataGrid(
        data_matrix=train_set['weather'].T,
        grid_points=t,
    ).to_basis(fourier_basis)

    linear_reg = skfda.ml.regression.LinearRegression(
    coef_basis=[fourier_basis])

    _ = linear_reg.fit(X_train, Y_train)

    return linear_reg

def performance_mid_term(test_set, fd_linear_reg):
    t = np.linspace(0, 23, 24)
    fourier_basis = skfda.representation.basis.FourierBasis(domain_range=(0,23), n_basis=3, period=24)
    X_test = skfda.FDataGrid(
        data_matrix=test_set['weather'].T,
        grid_points=t,
    ).to_basis(fourier_basis)
    Y_test = skfda.FDataGrid(
        data_matrix=test_set['temp'].T,
        grid_points=t,
    ).to_basis(fourier_basis)
    error = np.mean(np.abs(np.squeeze(fd_linear_reg.predict(X_test)(t)) - np.squeeze(Y_test(t))))
    return error

def evaluation_mid_term(all_sets, start_date):
    t = np.linspace(0, 23, 24)
    fourier_basis = skfda.representation.basis.FourierBasis(domain_range=(0,23), n_basis=3, period=24)

    Y_train = skfda.FDataGrid(
        data_matrix=all_sets['temp'].T,
        grid_points=t,
    ).to_basis(fourier_basis)
    X_train = skfda.FDataGrid(
        data_matrix=all_sets['weather'].T,
        grid_points=t,
    ).to_basis(fourier_basis)

    linear_reg = skfda.ml.regression.LinearRegression(
    coef_basis=[fourier_basis])

    _ = linear_reg.fit(X_train, Y_train)

    eval = np.squeeze(linear_reg.predict(X_train)(t)).T
    eval_Y_train = np.squeeze(Y_train(t)).T

    vector_recon = eval.flatten(order='F')
    vector_local = eval_Y_train.flatten(order='F')
    vector_weather = all_sets['weather'].to_numpy().flatten(order='F')
    dates = pd.date_range(start=start_date, periods=len(vector_recon), freq='h')

    return pd.DataFrame({'date': dates, 'temp_recon': vector_recon, 'temp_local': vector_local, 'temp_weather' : vector_weather})

def evaluation_mid_term_2(linear_reg, X):
    t = np.linspace(0, 23, 24)
    fourier_basis = skfda.representation.basis.FourierBasis(domain_range=(0,23), n_basis=3, period=24)

    results, start_date = transformation_of_data_2(X)

    X_fda = skfda.FDataGrid(
        data_matrix=results['weather'].T,
        grid_points=t,
    ).to_basis(fourier_basis)

    eval = np.squeeze(linear_reg.predict(X_fda)(t)).T

    vector_recon = eval.flatten(order='F')
    dates = pd.date_range(start=start_date, periods=len(vector_recon), freq='h')

    return pd.DataFrame({'date': dates, 'temp_recon': vector_recon})

In [ ]:
# @title Upload data

print('\n')
print('________________________')
print('Upload crime scene data.')
print('________________________')
print('\n')
uploaded = files.upload()
filename = list(uploaded.keys())[0]

scene_temp = pd.read_csv(filename, sep=';')

print('\n')
print('________________________')
print('Upload weather station data.')
print('________________________')
print('\n')
uploaded = files.upload()
filename = list(uploaded.keys())[0]

weather_temp = pd.read_csv(filename, sep=';')

In [ ]:
# @title Indicate columns

print('\n')
print('________________________')
print('Crime scene dataset:')
print('\n')
print('Columns:')
print(scene_temp.columns)
print('\n')
print('Provide the exact name of the column with temperature measurements.')
print('________________________')
print('\n')

original_name = input()
while original_name not in scene_temp.columns:
  print('Error! There is not such column in the dataset.')
  print('Provide the exact name of the column with temperature measurements.')
  print('\n')
  original_name = input()

scene_temp.rename(columns = {original_name: 'Y'}, inplace = True)

print('________________________')
print('Crime scene dataset:')
print('\n')
print('Columns:')
print(scene_temp.columns)
print('\n')
print('Provide the exact name of the column with weather station data.')
print('________________________')
print('\n')

original_name = input()
while original_name not in scene_temp.columns:
  print('Error! There is not such column in the dataset.')
  print('Provide the exact name of the column with temperature from weather station.')
  print('\n')
  original_name = input()

scene_temp.rename(columns = {original_name: 'X'}, inplace = True)

print('\n')
print('________________________')
print('Crime scene dataset:')
print('\n')
print('Columns:')
print(scene_temp.columns)
print('\n')
print('Provide the exact name of the column with date.')
print('________________________')
print('\n')

original_name = input()
while original_name not in scene_temp.columns:
  print('Error! There is not such column in the dataset.')
  print('Provide the exact name of the column with date.')
  print('\n')
  original_name = input()

scene_temp.rename(columns = {original_name: 'date'}, inplace = True)

print('\n')
print('________________________')
print('Crime scene dataset:')
print('\n')
print('Columns:')
print(scene_temp.columns)
print('\n')
print('Provide the exact name of the column with time.')
print('________________________')
print('\n')

original_name = input()
while original_name not in scene_temp.columns:
  print('Error! There is not such column in the dataset.')
  print('Provide the exact name of the column with time.')
  print('\n')
  original_name = input()

scene_temp.rename(columns = {original_name: 'time'}, inplace = True)

print('\n')
print('________________________')
print('Weather station dataset:')
print('\n')
print('Columns:')
print(weather_temp.columns)
print('\n')
print('Provide the exact name of the column with temperatures from weather station.')
print('________________________')
print('\n')

original_name = input()
while original_name not in weather_temp.columns:
  print('Error! There is not such column in the dataset.')
  print('Provide the exact name of the column with temperature from weather station.')
  print('\n')
  original_name = input()

weather_temp.rename(columns = {original_name: 'X'}, inplace = True)

print('\n')
print('________________________')
print('Weather station dataset:')
print('\n')
print('Columns:')
print(weather_temp.columns)
print('\n')
print('Provide the exact name of the column with date.')
print('________________________')
print('\n')

original_name = input()
while original_name not in weather_temp.columns:
  print('Error! There is not such column in the dataset.')
  print('Provide the exact name of the column with date.')
  print('\n')
  original_name = input()

weather_temp.rename(columns = {original_name: 'date'}, inplace = True)

print('\n')
print('________________________')
print('Weather station dataset:')
print('\n')
print('Columns:')
print(weather_temp.columns)
print('\n')
print('Provide the exact name of the column with time.')
print('________________________')
print('\n')

original_name = input()
while original_name not in weather_temp.columns:
  print('Error! There is not such column in the dataset.')
  print('Provide the exact name of the column with time.')
  print('\n')
  original_name = input()

weather_temp.rename(columns = {original_name: 'time'}, inplace = True)

scene_temp["datetime"] = scene_temp["date"] + " " + scene_temp["time"].astype(str) + ":00"
weather_temp["datetime"] = weather_temp["date"] + " " + weather_temp["time"].astype(str) + ":00"

scene_temp["datetime"] = pd.to_datetime(scene_temp["datetime"], format="%d.%m.%Y %H:%M")
weather_temp["datetime"] = pd.to_datetime(weather_temp["datetime"], format="%d.%m.%Y %H:%M")

scene_temp.drop(columns=["date", "time"], inplace=True)
weather_temp.drop(columns=["date", "time"], inplace=True)

scene_temp = scene_temp[["datetime"] + [col for col in scene_temp.columns if col != "datetime"]]
weather_temp = weather_temp[["datetime"] + [col for col in weather_temp.columns if col != "datetime"]]

scene_temp = scene_temp.replace(',','.', regex=True)
weather_temp = weather_temp.replace(',','.', regex=True)

scene_temp[['X', 'Y']] = scene_temp[['X', 'Y']].apply(pd.to_numeric, errors="coerce")
weather_temp[['X']] = weather_temp[['X']].apply(pd.to_numeric, errors="coerce")

In [ ]:
# @title Show crime scene data
scene_temp

In [ ]:
# @title Show weather station data
weather_temp

In [ ]:
# @title Evaluation of reconstruction
data = scene_temp.loc[:,["datetime",'Y','X']]
data.columns = ["date", "Y", "X"]
all_sets, train_sets, test_sets, start_date = transformation_of_data(data, n = 3)
fd_linear_reg = reconstruction_mid_term(train_sets)
score = performance_mid_term(test_sets, fd_linear_reg)
print(str(np.round(1)) + ': ' + str(score))

results = evaluation_mid_term(all_sets, start_date)
plt.figure(figsize=(10, 6))

plt.plot(results['date'], results['temp_weather'], label='temp_weather', color = 'blue')
plt.plot(results['date'], results['temp_local'], label='temp_local', color = 'black')
plt.plot(results['date'], results['temp_recon'], label='temp_recon', color = 'red')

plt.xlabel('Date')
plt.ylabel('Temperatures')
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.show()

In [11]:
# @title Reconstruction
weather1 = weather_temp.loc[:,['datetime','X']]
weather1.columns = ["date", "X"]
reconstruction = evaluation_mid_term_2(fd_linear_reg, weather1)
reconstruction

In [13]:
# @title Save reconstruction
reconstruction['time'] = reconstruction['date'].dt.hour
reconstruction['date'] = reconstruction['date'].dt.date
reconstruction = reconstruction[['date', 'time', 'temp_recon']]
reconstruction.to_csv('reconstruction.csv', sep = ';')

In [ ]:
# @title Delete all files
import shutil
import os

shutil.rmtree('/content/')
print("Environment restared.")